# Stage 4 — Base ASAG Dependency Parsing Pipeline

**Project:** Automatic Short Answer Grading using Semantic and Dependency Analysis  
**Repository:** `proposition-level-asag`  
**Current Stage:** Stage 4 — Base ASAG Dependency Parsing

---

### Purpose
This notebook demonstrates the **Stage 4 Dependency Parsing Component** for the Base ASAG system.

Building directly on Stage 3 Preprocessing, the dependency parser uses spaCy's dependency parser (`en_core_web_sm`) to establish the grammatical relationship between words:
- **Dependency Labels:** What grammatical role does each token play (e.g. `nsubj`, `dobj`, `prep`, `amod`, `ROOT`)?
- **Syntactic Heads:** What token is the governing parent of each dependent token?
- **Sentence-Relative Indexing:** Head indices point directly to positions within the sentence.

> **CRITICAL SEPARATION RULE:**  
> Stage 4 is strictly responsible for syntactic dependency parsing. It does **not** construct dependency triples (reserved for Stage 5), nor does it perform semantic matching, entailment, or grading.

> **NOTICE: DEMO DATA — NOT RESEARCH DATA**  
> The examples used here are synthetic demonstration examples for pipeline verification only.

## 1. Imports and Setup

We import pandas for tabular formatting and `DependencyParser` directly from `base_asag.dependency`. We also import `TextPreprocessor` from `base_asag.preprocessing` to demonstrate how Stage 4 seamlessly consumes Stage 3 representations.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure repository root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from base_asag.preprocessing import TextPreprocessor
from base_asag.dependency import DependencyParser, ParsedDocument, ParsedSentence

preprocessor = TextPreprocessor(model_name="en_core_web_sm")
parser = DependencyParser(model_name="en_core_web_sm")

print("Initialized TextPreprocessor and DependencyParser successfully.")
print(f"Model: {parser.model_name}")

Initialized TextPreprocessor and DependencyParser successfully.
Model: en_core_web_sm


## 2. Parse Reference Answer & Student Answer

We parse the global primary key demonstration example:
- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*
- **Student Answer:** *"A primary key uniquely identifies every row in a table."*

We demonstrate Stage 3 $\rightarrow$ Stage 4 pipeline chaining: `preprocessor.process(text)` $\rightarrow$ `parser.parse_sentence(processed_sent)`.

In [2]:
reference_text = "A primary key uniquely identifies each record in a table."
student_text = "A primary key uniquely identifies every row in a table."

# Preprocess via Stage 3
ref_proc = preprocessor.process(reference_text)
stu_proc = preprocessor.process(student_text)

# Parse via Stage 4
ref_parsed = parser.parse_sentence(ref_proc.sentences[0])
stu_parsed = parser.parse_sentence(stu_proc.sentences[0])

print(f'Reference Answer: "{ref_parsed.original_text}"')
print(f"  Root: {ref_parsed.root.text} [pos={ref_parsed.root.pos}, dep={ref_parsed.root.dep}]")
print()
print(f'Student Answer:   "{stu_parsed.original_text}"')
print(f"  Root: {stu_parsed.root.text} [pos={stu_parsed.root.pos}, dep={stu_parsed.root.dep}]")

Reference Answer: "A primary key uniquely identifies each record in a table."
  Root: identifies [pos=VERB, dep=ROOT]

Student Answer:   "A primary key uniquely identifies every row in a table."
  Root: identifies [pos=VERB, dep=ROOT]


## 3. Reference Answer Dependency Structure Table

Below is the parsed dependency table for the reference answer:
- **Index:** 0-based sentence position.
- **Token:** Surface string.
- **Lemma:** Lemmatized form.
- **POS:** Universal POS.
- **Dependency:** Grammatical relation label (e.g., `nsubj`, `dobj`, `ROOT`).
- **Head Text:** The governing parent token.
- **Head Index:** Sentence-relative index of the governing token.
- **Children:** Dependent token indices.

In [3]:
def parsed_sentence_to_dataframe(parsed_sent):
    rows = []
    for tok in parsed_sent.tokens:
        rows.append({
            "Index": tok.index,
            "Token": tok.text,
            "Lemma": tok.lemma,
            "POS": tok.pos,
            "Dependency": tok.dep,
            "Head Text": tok.head_text,
            "Head Index": tok.head_index,
            "Children": list(tok.children_indices),
        })
    return pd.DataFrame(rows)

print("--- Reference Answer Dependency Structure ---")
df_ref = parsed_sentence_to_dataframe(ref_parsed)
df_ref

--- Reference Answer Dependency Structure ---


,Index,Token,Lemma,POS,Dependency,Head Text,Head Index,Children
0,0,A,a,DET,det,key,2,[]
1,1,primary,primary,ADJ,amod,key,2,[]
2,2,key,key,NOUN,nsubj,identifies,4,"[0, 1]"
3,3,uniquely,uniquely,ADV,advmod,identifies,4,[]
4,4,identifies,identify,VERB,ROOT,identifies,4,"[2, 3, 6, 7, 10]"
5,5,each,each,DET,det,record,6,[]
6,6,record,record,NOUN,dobj,identifies,4,[5]
7,7,in,in,ADP,prep,identifies,4,[9]
8,8,a,a,DET,det,table,9,[]
9,9,table,table,NOUN,pobj,in,7,[8]


## 4. Student Answer Dependency Structure Table

In [4]:
print("--- Student Answer Dependency Structure ---")
df_stu = parsed_sentence_to_dataframe(stu_parsed)
df_stu

--- Student Answer Dependency Structure ---


,Index,Token,Lemma,POS,Dependency,Head Text,Head Index,Children
0,0,A,a,DET,det,key,2,[]
1,1,primary,primary,ADJ,amod,key,2,[]
2,2,key,key,NOUN,nsubj,identifies,4,"[0, 1]"
3,3,uniquely,uniquely,ADV,advmod,identifies,4,[]
4,4,identifies,identify,VERB,ROOT,identifies,4,"[2, 3, 6, 7, 10]"
5,5,every,every,DET,det,row,6,[]
6,6,row,row,NOUN,dobj,identifies,4,[5]
7,7,in,in,ADP,prep,identifies,4,[9]
8,8,a,a,DET,det,table,9,[]
9,9,table,table,NOUN,pobj,in,7,[8]


## 5. Textual Dependency Tree Visualization

To clearly visualize the syntactic hierarchy, we display an indented textual tree rooted at the sentence ROOT (`identifies`).

In [5]:
def print_dependency_tree(parsed_sent, token=None, prefix="", is_last=True):
    if token is None:
        token = parsed_sent.root
        print(f"ROOT: {token.text} [{token.pos}] (idx={token.index})")
        children = [parsed_sent.get_token(i) for i in token.children_indices]
        for idx, child in enumerate(children):
            print_dependency_tree(parsed_sent, child, "  ", idx == len(children) - 1)
        return

    connector = "└── " if is_last else "├── "
    print(f"{prefix}{connector}{token.dep.upper()}: {token.text} [{token.pos}] (idx={token.index}, head={token.head_text})")
    
    child_prefix = prefix + ("    " if is_last else "│   ")
    children = [parsed_sent.get_token(i) for i in token.children_indices]
    for idx, child in enumerate(children):
        print_dependency_tree(parsed_sent, child, child_prefix, idx == len(children) - 1)

print("=== Reference Answer Dependency Tree ===")
print_dependency_tree(ref_parsed)
print("\n=== Student Answer Dependency Tree ===")
print_dependency_tree(stu_parsed)

=== Reference Answer Dependency Tree ===
ROOT: identifies [VERB] (idx=4)
  ├── NSUBJ: key [NOUN] (idx=2, head=identifies)
  │   ├── DET: A [DET] (idx=0, head=key)
  │   └── AMOD: primary [ADJ] (idx=1, head=key)
  ├── ADVMOD: uniquely [ADV] (idx=3, head=identifies)
  ├── DOBJ: record [NOUN] (idx=6, head=identifies)
  │   └── DET: each [DET] (idx=5, head=record)
  ├── PREP: in [ADP] (idx=7, head=identifies)
  │   └── POBJ: table [NOUN] (idx=9, head=in)
  │       └── DET: a [DET] (idx=8, head=table)
  └── PUNCT: . [PUNCT] (idx=10, head=identifies)

=== Student Answer Dependency Tree ===
ROOT: identifies [VERB] (idx=4)
  ├── NSUBJ: key [NOUN] (idx=2, head=identifies)
  │   ├── DET: A [DET] (idx=0, head=key)
  │   └── AMOD: primary [ADJ] (idx=1, head=key)
  ├── ADVMOD: uniquely [ADV] (idx=3, head=identifies)
  ├── DOBJ: row [NOUN] (idx=6, head=identifies)
  │   └── DET: every [DET] (idx=5, head=row)
  ├── PREP: in [ADP] (idx=7, head=identifies)
  │   └── POBJ: table [NOUN] (idx=9, head=in)


## 6. Syntactic Comparison & Readiness for Stage 5

Notice the structural isomorphism between the reference and student answers:
- Both sentences have the identical root predicate: `identifies` (`VERB`).
- Both share the nominal subject subtree: `key` $\leftarrow$ `primary`, `A`.
- In the reference answer, the direct object is `record` (with determiner `each`).
- In the student answer, the direct object is `row` (with determiner `every`).
- Both attach the prepositional phrase `in a table`.

In **Stage 5 (Dependency Triples)**, these syntactic relations will be compiled into canonical triples such as `(key, identifies, record)` and `(key, identifies, row)`.

## 7. Summary

Stage 4 Dependency Parsing is complete:
- [x] Dependency parsing enabled using spaCy `en_core_web_sm`.
- [x] Full dependency labels (`nsubj`, `dobj`, `prep`, `pobj`, `ROOT`) extracted.
- [x] Valid sentence-relative head indexing.
- [x] Punctuation, stop words, and negations fully preserved.
- [x] No dependency triples generated yet (strictly scoped for Stage 5).